# Principles of Natural Language Processing Lab  
## Language Modelling: N-grams, N-gram Probabilities, Evaluation and Perplexity

**Course:** AME 5053 — Principles of Natural Language Processing Lab  
**Lab duration:** 3 hours  
**Mode:** Guided implementation + comparison + interpretation

### Learning outcomes
By the end of this lab, you should be able to:

1. construct unigram, bigram and trigram language models from a text corpus;
2. estimate N-gram probabilities using maximum-likelihood estimation (MLE);
3. compute the probability of a sentence under an N-gram model;
4. explain the effect of sentence-boundary symbols such as `<s>` and `</s>`;
5. evaluate a language model on unseen text using perplexity;
6. diagnose why an unsmoothed N-gram model can fail on unseen N-grams.

### What you must submit
Your notebook should contain:
- your predictions before running selected cells;
- completed implementations;
- at least one comparison between models;
- an error analysis using concrete examples;
- a short conclusion explaining what model behaviour you observed.

## 1. Corpus used in this lab

We will use a small, controlled corpus so that the counts can be inspected manually.

```text
students learn natural language processing
students learn machine learning
students study language models
language models predict words
language models assign probabilities
machine learning models learn patterns
natural language processing uses language models
```

The corpus is intentionally small. This makes the probability calculations transparent, but it will also expose an important limitation: **many valid word sequences will never occur in the training corpus**.

We will later split examples into **training** and **test** sentences so that perplexity is computed on data that the model did not simply memorize.

In [2]:
from collections import Counter, defaultdict
import math
import pandas as pd

raw_sentences = [
    "students learn natural language processing",
    "students learn machine learning",
    "students study language models",
    "language models predict words",
    "language models assign probabilities",
    "machine learning models learn patterns",
    "natural language processing uses language models",
]

raw_sentences

['students learn natural language processing',
 'students learn machine learning',
 'students study language models',
 'language models predict words',
 'language models assign probabilities',
 'machine learning models learn patterns',
 'natural language processing uses language models']

## 1.1 Tokenize the corpus
Run the helper below and inspect the output.

In [3]:
def tokenize(sentence):
    return sentence.lower().split()

tokenized = [tokenize(s) for s in raw_sentences]
tokenized

[['students', 'learn', 'natural', 'language', 'processing'],
 ['students', 'learn', 'machine', 'learning'],
 ['students', 'study', 'language', 'models'],
 ['language', 'models', 'predict', 'words'],
 ['language', 'models', 'assign', 'probabilities'],
 ['machine', 'learning', 'models', 'learn', 'patterns'],
 ['natural', 'language', 'processing', 'uses', 'language', 'models']]

## 2. Sentence boundaries

A language model should know not only which words can follow other words, but also where a sentence can begin and end.

For a bigram model we will represent:

```text
students learn machine learning
```

as

```text
<s> students learn machine learning </s>
```

For a trigram model, we need enough history at the beginning. We therefore use two start symbols:

```text
<s> <s> students learn machine learning </s>
```

This convention allows the model to estimate probabilities for sentence beginnings as well as sentence endings.

### Prediction 1 — before coding

Without running any code, answer:

1. Which word do you expect to be the most frequent unigram?
2. Which bigram do you expect to have a relatively high probability?
3. Do you expect a trigram model to assign non-zero probability to more or fewer unseen test sentences than a bigram model? Why?

Write your answers below.

**Your prediction:**  
- Most frequent unigram:  
- Likely high-probability bigram:  
- Bigram vs trigram on unseen text:

## 3. Build N-gram counts

For an N-gram model, we count neighbouring sequences of words.

For example, in:

```text
language models predict words
```

the bigrams are:

```text
(<s>, language)
(language, models)
(models, predict)
(predict, words)
(words, </s>)
```

The trigram model uses sequences of three tokens.

We will use these counts to estimate conditional probabilities.

In [4]:
def add_boundaries(tokens, n):
    # TODO:
    # Bigram model: add one <s> and one </s>
    # Trigram model: add two <s> symbols and one </s>
    if n == 1:
      return list(tokens)
    return ["<s>"] * (n-1) + list(tokens) + ["</s>"]

print(add_boundaries(["students", "learn"], 2))
print(add_boundaries(["students", "learn"], 3))

['<s>', 'students', 'learn', '</s>']
['<s>', '<s>', 'students', 'learn', '</s>']


In [5]:
def make_ngrams(tokens, n):
    # TODO: return all adjacent n-grams as tuples
    # Example: ['a','b','c'], n=2 -> [('a','b'), ('b','c')]
  return [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]

print(make_ngrams(["a", "b", "c"], 1))
print(make_ngrams(["a", "b", "c"], 2))
print(make_ngrams(["a", "b", "c"], 3))

[('a',), ('b',), ('c',)]
[('a', 'b'), ('b', 'c')]
[('a', 'b', 'c')]


In [7]:
# TODO: build unigram, bigram and trigram counts over the whole corpus.
# Suggested names:
# unigram_counts
# bigram_counts
# trigram_counts

unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for sent in tokenized:
  unigram_counts.update(sent)
  bigram_counts.update(make_ngrams(add_boundaries(sent, 2), 2))
  trigram_counts.update(make_ngrams(add_boundaries(sent, 3), 3))

print("Unigram types : ", len(unigram_counts))
print("Bigram types : ", len(bigram_counts))
print("Trigram types : ", len(trigram_counts))

Unigram types :  15
Bigram types :  28
Trigram types :  32


In [8]:
unigram_counts

Counter({'students': 3,
         'learn': 3,
         'natural': 2,
         'language': 6,
         'processing': 2,
         'machine': 2,
         'learning': 2,
         'study': 1,
         'models': 5,
         'predict': 1,
         'words': 1,
         'assign': 1,
         'probabilities': 1,
         'patterns': 1,
         'uses': 1})

In [9]:
bigram_counts

Counter({('<s>', 'students'): 3,
         ('students', 'learn'): 2,
         ('learn', 'natural'): 1,
         ('natural', 'language'): 2,
         ('language', 'processing'): 2,
         ('processing', '</s>'): 1,
         ('learn', 'machine'): 1,
         ('machine', 'learning'): 2,
         ('learning', '</s>'): 1,
         ('students', 'study'): 1,
         ('study', 'language'): 1,
         ('language', 'models'): 4,
         ('models', '</s>'): 2,
         ('<s>', 'language'): 2,
         ('models', 'predict'): 1,
         ('predict', 'words'): 1,
         ('words', '</s>'): 1,
         ('models', 'assign'): 1,
         ('assign', 'probabilities'): 1,
         ('probabilities', '</s>'): 1,
         ('<s>', 'machine'): 1,
         ('learning', 'models'): 1,
         ('models', 'learn'): 1,
         ('learn', 'patterns'): 1,
         ('patterns', '</s>'): 1,
         ('<s>', 'natural'): 1,
         ('processing', 'uses'): 1,
         ('uses', 'language'): 1})

In [10]:
trigram_counts

Counter({('<s>', '<s>', 'students'): 3,
         ('<s>', 'students', 'learn'): 2,
         ('students', 'learn', 'natural'): 1,
         ('learn', 'natural', 'language'): 1,
         ('natural', 'language', 'processing'): 2,
         ('language', 'processing', '</s>'): 1,
         ('students', 'learn', 'machine'): 1,
         ('learn', 'machine', 'learning'): 1,
         ('machine', 'learning', '</s>'): 1,
         ('<s>', 'students', 'study'): 1,
         ('students', 'study', 'language'): 1,
         ('study', 'language', 'models'): 1,
         ('language', 'models', '</s>'): 2,
         ('<s>', '<s>', 'language'): 2,
         ('<s>', 'language', 'models'): 2,
         ('language', 'models', 'predict'): 1,
         ('models', 'predict', 'words'): 1,
         ('predict', 'words', '</s>'): 1,
         ('language', 'models', 'assign'): 1,
         ('models', 'assign', 'probabilities'): 1,
         ('assign', 'probabilities', '</s>'): 1,
         ('<s>', '<s>', 'machine'): 1,
         ('

### Inspect the most frequent N-grams
Create tables showing the 10 most frequent unigrams, bigrams and trigrams.

In [11]:
# TODO: display the 10 most frequent items from each Counter.
def top_table(counter, k = 10):
  return pd.DataFrame(counter.most_common(k), columns = ["N-gram", "Count"])

display(top_table(unigram_counts))
display(top_table(bigram_counts))
display(top_table(trigram_counts))

,N-gram,Count
0,language,6
1,models,5
2,students,3
3,learn,3
4,natural,2
5,processing,2
6,machine,2
7,learning,2
8,study,1
9,predict,1


,N-gram,Count
0,"(language, models)",4
1,"(<s>, students)",3
2,"(students, learn)",2
3,"(natural, language)",2
4,"(language, processing)",2
5,"(machine, learning)",2
6,"(models, </s>)",2
7,"(<s>, language)",2
8,"(learn, natural)",1
9,"(processing, </s>)",1


,N-gram,Count
0,"(<s>, <s>, students)",3
1,"(<s>, students, learn)",2
2,"(natural, language, processing)",2
3,"(language, models, </s>)",2
4,"(<s>, <s>, language)",2
5,"(<s>, language, models)",2
6,"(students, learn, natural)",1
7,"(learn, natural, language)",1
8,"(language, processing, </s>)",1
9,"(students, learn, machine)",1


## 4. Maximum-likelihood estimation of N-gram probabilities

For a bigram model,

$$
P(w_i \mid w_{i-1})
=
\frac{C(w_{i-1}, w_i)}{C(w_{i-1})}
$$

For a trigram model,

$$
P(w_i \mid w_{i-2}, w_{i-1})
=
\frac{C(w_{i-2}, w_{i-1}, w_i)}
     {C(w_{i-2}, w_{i-1})}
$$

The key idea is simple: **among all occurrences of the history, how often was the next word the one we are asking about?**

This is a maximum-likelihood estimate because the probabilities are obtained directly from observed frequencies in the training corpus.

In [13]:
bigram_context_counts = Counter()
for (w1, w2), c in bigram_counts.items():
      bigram_context_counts[w1] += c

trigram_context_counts = Counter()
for (w1, w2, w3), c in trigram_counts.items():
  trigram_context_counts[(w1, w2)] += c

def bigram_probability(w1, w2, bigram_counts, unigram_context_counts):
    # TODO: implement P(w2 | w1)
    denominator = unigram_context_counts[w1]
    if denominator == 0:
      return 0.0
    return bigram_counts[(w1, w2)] / denominator

def trigram_probability(w1, w2, w3, trigram_counts, bigram_context_counts):
    # TODO: implement P(w3 | w1,w2)
    denominator  = bigram_context_counts[(w1, w2)]
    if denominator == 0:
      return 0.0
    return trigram_counts[(w1, w2, w3)] / denominator

### Probability checks
Compute and interpret the following:

- `P(learn | students)`
- `P(models | language)`
- `P(language | learn)`
- `P(models | study, language)`

In [14]:
# TODO: create the context counts needed by the probability functions,
# then compute the four probabilities above.

checks = {
    "P(learn | students)": bigram_probability("students","learn", bigram_counts, bigram_context_counts),
    "P(models | language)": bigram_probability("language","models", bigram_counts, bigram_context_counts),
    "P(language | learn)": bigram_probability("learn","language", bigram_counts, bigram_context_counts),
    "P(models | study, language)": trigram_probability("study","language","models", trigram_counts, trigram_context_counts),
}

for k, v in checks.items():
    print(f"{k:32s} = {v:.4f}")


P(learn | students)              = 0.6667
P(models | language)             = 0.6667
P(language | learn)              = 0.0000
P(models | study, language)      = 1.0000


## 5. Sentence probability

Under a bigram model, the probability of a sentence is approximated using the Markov assumption:

$$
P(w_1,\ldots,w_m)
\approx
P(w_1\mid <s>)
\prod_{i=2}^{m}P(w_i\mid w_{i-1})
P(</s>\mid w_m)
$$

The trigram model conditions each word on the previous two tokens.

Because sentence probabilities are products of many values smaller than 1, they can become extremely small. In larger systems we therefore often work with **log probabilities**.

In [ ]:
def bigram_sentence_probability(sentence, bigram_counts, context_counts):
    # TODO:
    # 1. tokenize
    # 2. add boundaries
    tokens = add_boundaries(tokenize(sentence), 2)
    # 3. multiply bigram probabilities
    prob = 1.0
    for w1, w2 in make_ngrams(tokens, 2):
      prob *= bigram_probability(w1, w2, bigram_counts, context_counts)
    return prob

def trigram_sentence_probability(sentence, trigram_counts, context_counts):
    # TODO:
    # 1. tokenize
    # 2. add trigram boundaries
    tokens = add_boundaries(tokenize(sentence), 3)
    # 3. multiply trigram probabilities
    prob = 1.0
    for w1, w2, w3 in make_ngrams(tokens, 3):
      prob *= trigram_probability(w1, w2, w3,  trigram_counts, context_counts)
    return prob

### Compare sentence probabilities
Evaluate these two training-like sentences:

- `students learn machine learning`
- `language models assign probabilities`

Then explain why their probabilities differ.

In [ ]:
# TODO: compute bigram and trigram sentence probabilities for the two sentences.
for s in [
    "students learn machine learning",
    "language models assign probabilities"
]:
  print("\n",s)
  print("Bigram probability : ", bigram_sentence_probability(s, bigram_counts, bigram_context_counts))
  print("Trigram probability : ", trigram_sentence_probability(s, trigram_counts, trigram_context_counts))


 students learn machine learning
Bigram probability :  0.047619047619047616
Trigram probability :  0.07142857142857142

 language models assign probabilities
Bigram probability :  0.0380952380952381
Trigram probability :  0.07142857142857142


## 6. Evaluation on unseen text

A useful language model should perform well on text that was **not used to estimate its probabilities**.

We will use:

### Test sentence A
`students learn language models`

### Test sentence B
`language models learn patterns`

### Test sentence C
`students predict probabilities`

Before computing anything, inspect the corpus and predict which test sentences may contain unseen bigrams or trigrams.

### Prediction 2 — unseen sequences

Before running the next section, list at least one bigram/trigram that you think is unseen in each test sentence.

**A. students learn language models**  
Prediction:

**B. language models learn patterns**  
Prediction:

**C. students predict probabilities**  
Prediction:

## 7. Perplexity

Perplexity is a standard intrinsic evaluation measure for language models.

For a sequence containing \(N\) predicted tokens,

$$
PP(W)
=
P(W)^{-1/N}
$$

Equivalently, using log probabilities,

$$
PP(W)
=
\exp\left(
-\frac{1}{N}
\sum_{i=1}^{N}\log P(w_i \mid \text{history})
\right)
$$

Interpretation:

- **lower perplexity** means the model assigns higher probability to the observed test sequence;
- **higher perplexity** means the sequence is more surprising to the model;
- if an unsmoothed model encounters an N-gram with probability zero, the perplexity becomes infinite.

Perplexity should only be compared meaningfully when models are evaluated on the **same tokenization and same test data**.

In [21]:
def bigram_perplexity(sentence, bigram_counts, context_counts):
    # TODO:
    # Use log probabilities.
    # If any required bigram has probability 0, return math.inf.
    tokens = add_boundaries(tokenize(sentence), 2)
    grams = make_ngrams(tokens, 2)
    log_prob_sum = 0.0

    for w1, w2 in grams:
      p = bigram_probability(w1, w2, bigram_counts, context_counts)
      if p == 0:
        return math.inf
      log_prob_sum += math.log(p)

    N = len(grams)
    return math.exp(-log_prob_sum / N)


def trigram_perplexity(sentence, trigram_counts, context_counts):
    # TODO:
    # Use log probabilities.
    # If any required trigram has probability 0, return math.inf.
    tokens = add_boundaries(tokenize(sentence), 3)
    grams = make_ngrams(tokens, 3)
    log_prob_sum = 0.0

    for w1, w2, w3 in grams:
      p = trigram_probability(w1, w2, w3, trigram_counts, context_counts)
      if p == 0:
        return math.inf
      log_prob_sum += math.log(p)

    N = len(grams)
    return math.exp(-log_prob_sum / N)

In [22]:
test_sentences = [
    "students learn language models",
    "language models learn patterns",
    "students predict probabilities",
]

# TODO: construct a DataFrame with:
# sentence | bigram_perplexity | trigram_perplexity
rows = []
for s in test_sentences:
  rows.append({
      "sentence" : s,
      "bigram_perplexity" : bigram_perplexity(s, bigram_counts, bigram_context_counts),
      "trigram perplexity" : trigram_perplexity(s, trigram_counts, trigram_context_counts),

  })

results = pd.DataFrame(rows)
results

,sentence,bigram_perplexity,trigram perplexity
0,students learn language models,inf,inf
1,language models learn patterns,2.394694,inf
2,students predict probabilities,inf,inf


## 8. Diagnose the model
For every test sentence with infinite perplexity, print the first unseen N-gram that causes the failure.

In [26]:
def first_unseen_ngram(sentence, n, ngram_counts):
    # TODO: return the first unseen n-gram, or None if every n-gram was seen.
    tokens = add_boundaries(tokenize(sentence), n)
    for gram in make_ngrams(tokens, n):
      if ngram_counts[gram] == 0:
        return gram
    return None

# TODO: apply this to the test sentences for n=2 and n=3.
diagnostics = []
for s in test_sentences:
  diagnostics.append({
      "sentence" : s,
      "first unseen bigram" : first_unseen_ngram(s, 2, bigram_counts),
      "first unseen trigram" : first_unseen_ngram(s, 3, trigram_counts),
  })

pd.DataFrame(diagnostics)

,sentence,first unseen bigram,first unseen trigram
0,students learn language models,"(learn, language)","(students, learn, language)"
1,language models learn patterns,None,"(language, models, learn)"
2,students predict probabilities,"(students, predict)","(<s>, students, predict)"


### Expected interpretation

The lab should lead students to three conclusions:

1. **Higher-order context can be more informative.**  
   A trigram can capture distinctions that a bigram cannot.

2. **Higher-order models are more sparse.**  
   With limited training data, exact trigram histories are much less likely to have been observed.

3. **Perplexity exposes this directly.**  
   An unseen required N-gram in an unsmoothed model gives zero probability and therefore infinite perplexity.

Do not present “trigram is better than bigram” as a general rule. Model order, corpus size, smoothing and the target domain all matter.

## 9. Interpretation and error analysis

Answer in complete sentences.

1. Which model gave lower perplexity on the test sentences that both models could score?
2. Did the higher-order model always perform better? Explain using the size of this corpus.
3. Identify one unseen bigram and one unseen trigram from the test set.
4. Why does a zero probability create a serious problem when sentence probability is computed by multiplication?
5. What would you expect to happen if the training corpus became much larger?

### Suggested answers / points to look for

1. Students should compare only sentences for which both models return finite values; otherwise the comparison is dominated by zero probability.
2. No. On this tiny corpus, the trigram model is more sparse and can fail even when the bigram model has all required local transitions.
3. Examples should be taken from the diagnostic table rather than invented.
4. A single zero factor makes the product for the whole sentence zero.
5. More observed contexts should reduce sparsity, though the number of possible N-grams also grows rapidly.

## 10. Viva / discussion questions

1. Why is an N-gram language model called a probabilistic model?
2. What is the Markov assumption in a bigram model?
3. Why are sentence boundary symbols useful?
4. Why can a trigram model be more data-hungry than a bigram model?
5. Why is perplexity usually preferred over raw sentence probability when comparing sequences of different lengths?
6. Why can an unsmoothed N-gram model assign infinite perplexity to a perfectly grammatical sentence?